# Phase 5F — Gradient Boosting: XGBoost & LightGBM

Gradient boosting models are the **dominant algorithm for tabular data** in competitions and industry.

**Theory:** Boosting builds an ensemble sequentially. Each new model focuses on the mistakes of the previous ones. Uses gradient descent in function space.

**Install:** `pip install xgboost lightgbm scikit-learn`

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report
from sklearn.ensemble import GradientBoostingClassifier, AdaBoostClassifier
from sklearn.datasets import make_classification
from sklearn.preprocessing import LabelEncoder

np.random.seed(42)
sns.set_theme(style="whitegrid")

---
## 1. Boosting vs Bagging Intuition

In [ ]:
from sklearn.ensemble import RandomForestClassifier

X, y = make_classification(
    n_samples=1000, n_features=15, n_informative=8, n_redundant=4, random_state=42
)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

models = {
    "AdaBoost (boosting)": AdaBoostClassifier(n_estimators=100, random_state=42),
    "sklearn GBM (boosting)": GradientBoostingClassifier(
        n_estimators=100, max_depth=3, random_state=42
    ),
    "Random Forest (bagging)": RandomForestClassifier(
        n_estimators=100, random_state=42
    ),
}

print("=== Bagging vs Boosting ===")
for name, model in models.items():
    cv = cross_val_score(model, X, y, cv=5, scoring="accuracy")
    print(f"{name:35s}: {cv.mean():.4f} ± {cv.std():.4f}")

---
## 2. XGBoost

In [ ]:
try:
    import xgboost as xgb

    print(f"XGBoost version: {xgb.__version__}")

    # XGBoost — sklearn-compatible API
    xgb_model = xgb.XGBClassifier(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        use_label_encoder=False,
        eval_metric="logloss",
        random_state=42,
        verbosity=0,
    )
    xgb_model.fit(X_train, y_train)
    y_pred_xgb = xgb_model.predict(X_test)
    print(f"XGBoost accuracy: {accuracy_score(y_test, y_pred_xgb):.4f}")

    # Feature importance
    fig, ax = plt.subplots(figsize=(8, 5))
    feat_imp = pd.Series(xgb_model.feature_importances_).sort_values()
    feat_imp.plot(kind="barh", ax=ax, color="steelblue")
    ax.set_title("XGBoost Feature Importance")
    ax.set_xlabel("Importance")
    plt.tight_layout()
    plt.show()

except ImportError:
    print("XGBoost not installed. Run: pip install xgboost")
    print()
    print("Key XGBoost hyperparameters:")
    print("  n_estimators   : number of trees (more = better but slower)")
    print("  max_depth      : depth of each tree (3-6 typical)")
    print("  learning_rate  : step size (lower = more robust, needs more trees)")
    print("  subsample      : fraction of rows per tree (0.8 typical)")
    print("  colsample_bytree: fraction of columns per tree")
    print("  reg_alpha/lambda: L1/L2 regularization")

---
## 3. LightGBM — Faster Alternative

In [ ]:
try:
    import lightgbm as lgb

    print(f"LightGBM version: {lgb.__version__}")

    lgb_model = lgb.LGBMClassifier(
        n_estimators=200,
        num_leaves=31,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        verbose=-1,
    )
    lgb_model.fit(X_train, y_train)
    y_pred_lgb = lgb_model.predict(X_test)
    print(f"LightGBM accuracy: {accuracy_score(y_test, y_pred_lgb):.4f}")

except ImportError:
    print("LightGBM not installed. Run: pip install lightgbm")
    print()
    print("LightGBM vs XGBoost:")
    print("  LightGBM: faster, better on large datasets, leaf-wise growth")
    print("  XGBoost : slightly more robust, level-wise growth, more mature")
    print("  Both have near-identical sklearn API")

---
## 4. Hyperparameter Tuning with GridSearchCV

In [ ]:
# Using sklearn's GradientBoostingClassifier (no extra install needed)
from sklearn.ensemble import GradientBoostingClassifier

param_grid = {
    "n_estimators": [50, 100],
    "max_depth": [2, 3, 4],
    "learning_rate": [0.05, 0.1],
}

grid_search = GridSearchCV(
    GradientBoostingClassifier(random_state=42),
    param_grid,
    cv=3,
    scoring="accuracy",
    n_jobs=-1,
    verbose=0,
)
grid_search.fit(X_train, y_train)

print(f"Best parameters: {grid_search.best_params_}")
print(f"Best CV accuracy: {grid_search.best_score_:.4f}")

best_model = grid_search.best_estimator_
print(f"Test accuracy: {accuracy_score(y_test, best_model.predict(X_test)):.4f}")

---
## Summary

| Library | Speed | Memory | Large Data | Handles Missing |
|---------|-------|--------|------------|----------------|
| sklearn GBM | Slow | Medium | No | No |
| XGBoost | Fast | Medium | Yes | Yes (natively) |
| LightGBM | Fastest | Low | Yes | Yes (natively) |

**General advice for tabular data:**
1. Start with baseline (Logistic Regression)
2. Try Random Forest — robust, good defaults
3. Try XGBoost or LightGBM — usually best performance
4. Tune with RandomizedSearchCV (more efficient than GridSearch for many params)